# Создание и заполнение данных БД Postgre

In [400]:
%pip install python-dotenv psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [401]:
import os
import json
import psycopg2
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [402]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=zhora, DB=my_db_sviridov, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [403]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [404]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [405]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- user_logs
- departments
- enriched_user_logs


Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

s_all_avg, s_course_viewed_avg, s_q_attempt_viewed_avg, s_a_course_module_viewed_avg, s_a_submission_status_viewed_avg

In [406]:
#вывести среднюю оценку по курсу за которым закреплено наибольшее количество кафедр

cursor.execute("SELECT COUNT(DISTINCT Depart), courseid FROM user_logs; ")

row = cursor.fetchall()
print("\nТаблицы в базе данных:")
for row in row:
    print(row)

GroupingError: column "user_logs.courseid" must appear in the GROUP BY clause or be used in an aggregate function
LINE 1: SELECT COUNT(DISTINCT Depart), courseid FROM user_logs; 
                                       ^


## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [ ]:
cursor.execute("SELECT s_all_avg, s_course_viewed_avg, s_q_attempt_viewed_avg, s_a_course_module_viewed_avg, s_a_submission_status_viewed_avg FROM user_logs LIMIT 10;")

row = cursor.fetchall()
print("\nТаблицы в базе данных:")
for row in row:
    print(row)


Таблицы в базе данных:
(16.0, 9.0, 0.0, 0.0, 0.0)
(10.0, 6.0, 0.0, 0.0, 0.0)
(26.0, 15.0, 0.0, 0.0, 0.0)
(5.0, 1.0, 0.0, 0.0, 0.0)
(0.0, 0.0, 0.0, 0.0, 0.0)
(0.0, 0.0, 0.0, 0.0, 0.0)
(0.0, 0.0, 0.0, 0.0, 0.0)
(0.0, 0.0, 0.0, 0.0, 0.0)
(0.0, 0.0, 0.0, 0.0, 0.0)
(0.0, 0.0, 0.0, 0.0, 0.0)


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [ ]:
cursor.execute("SELECT COUNT(DISTINCT depart) FROM user_logs;")

row = cursor.fetchall()

for row in row:
    print(row)

(43,)


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [ ]:
cursor.execute("""
    SELECT d.department_name, COUNT(DISTINCT l.courseid) AS course_count
    FROM user_logs l
    JOIN departments d ON l.depart_id = d.id
    GROUP BY d.department_name
    ORDER BY course_count DESC;
""")
print("Кафедра | Количество курсов")
for row in cursor.fetchall():
    print(f"{row[0]}: {row[1]}")

Кафедра | Количество курсов
ДиСО: 53
ПОиД: 42
МиХТ: 42
ГМиТТК: 40
ЛиУТС: 36
БИиИТ: 35
ГМУиУП: 35
ПиЭММО: 33
АЭПиМ: 33
ЛиП: 33
ПиСЗ: 32
Эконом.: 32
ТОМ: 30
ГМДиОПИ: 29
ЛПиМ: 28
РМПИ: 28
МиТОДиМ: 28
ВТиП: 25
Психол.: 23
ВИ: 22
ТССА: 21
Дизайна: 20
Менеджм.: 20
ЯиЛ: 20
CC: 19
ПМиИ: 19
АиИИ: 19
ТиЭС: 19
Химии: 18
ХОМ: 18
РЯОЯиМК: 17
ЭПП: 16
ИиИБ: 16
АСУ: 16
Физики: 16
ЭиМЭ: 16
УиИС: 15
СРиППО: 14
ПЭиБЖД: 11
Физкульт.: 5
ЦДОМ: 4
ИТМ: 3
УСиБА: 2


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [ ]:
cursor.execute("" \
"SELECT COUNT(*)" \
"FROM (SELECT courseid FROM user_logs GROUP BY courseid HAVING COUNT(DISTINCT depart_id) > 1) AS multi")

multiCount = cursor.fetchone()[0]
print(f"Курсов с несколькими кафедрами = {multiCount}")


cursor.execute("SELECT l.courseid, STRING_AGG(DISTINCT d.department_name, ', ') AS departments FROM user_logs AS l " \
"JOIN departments d ON l.depart_id = d.id " \
"GROUP BY(l.courseid) " \
"HAVING COUNT(DISTINCT l.depart_id) > 1 " \
"ORDER BY l.courseid;")

print("\nКурсы и совместные кафедры:")
for row in cursor.fetchall():
    print(f"Курс {row[0]}: {row[1]}")

Курсов с несколькими кафедрами = 60

Курсы и совместные кафедры:
Курс 71495: ГМиТТК, ПиСЗ, УиИС
Курс 71508: ПиСЗ, УиИС
Курс 71541: ПиСЗ, УиИС
Курс 71547: ПиСЗ, УиИС
Курс 71549: ГМиТТК, ТиЭС
Курс 71571: ЛиУТС, ПиСЗ, УиИС
Курс 71632: CC, ВТиП
Курс 71736: ГМДиОПИ, ЭиМЭ
Курс 71852: АЭПиМ, ЭПП
Курс 71857: АЭПиМ, ЭПП
Курс 71884: АЭПиМ, ЭПП
Курс 71892: АЭПиМ, ТиЭС, ЭПП
Курс 71904: АЭПиМ, ЭПП
Курс 72126: АЭПиМ, УиИС
Курс 72314: Дизайна, ЛПиМ
Курс 72347: Дизайна, ЛПиМ
Курс 72358: МиХТ, ТОМ
Курс 72359: ЛПиМ, МиХТ, ТОМ
Курс 72380: ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 72392: ЛПиМ, МиХТ, ТОМ
Курс 72402: ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 72416: ЛПиМ, МиХТ, ТОМ
Курс 72447: ЛПиМ, МиХТ, ТОМ
Курс 72457: ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 72460: ЛПиМ, МиХТ, ТОМ
Курс 72462: МиХТ, ТОМ
Курс 72469: МиХТ, ТОМ
Курс 72800: АЭПиМ, ХОМ
Курс 72865: Дизайна, ТОМ
Курс 72885: Дизайна, ТОМ, ХОМ
Курс 73574: CC, ДиСО
Курс 73986: БИиИТ, Эконом.
Курс 75810: ГМДиОПИ, ГМиТТК, РМПИ
Курс 75833: ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ
Курс 75834: ГМДиОП

## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [ ]:
cursor.execute("""
    SELECT NameR_Level, COUNT(DISTINCT userid) AS studentCount
    FROM user_logs
    WHERE NameR_Level IN ('2','3','4','5')
    GROUP BY NameR_Level
    ORDER BY NameR_Level;
""")

print("| Оценка | Количество студентов |")
for row in cursor.fetchall():
    print(f"| {row[0]}       | {row[1]} |")

| Оценка | Количество студентов |
| 2       | 1069 |
| 3       | 1884 |
| 4       | 3243 |
| 5       | 3407 |


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

In [ ]:
cursor.execute("SELECT userid, SUM(s_all) AS total_logs FROM user_logs " \
"GROUP BY userid " \
"ORDER BY total_logs DESC " \
"LIMIT 1; ")
user_id, total = cursor.fetchone()
print(f"Студент {user_id} — {total} логов (максимум)")


Студент 28871 — 10300 логов (максимум)


## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [ ]:
cursor.execute("SELECT num_week, AVG(s_all) AS avg_events FROM user_logs " \
"GROUP BY num_week " \
"ORDER BY num_week; " \
"")

print("Неделя | Среднее количество событий")
for row in cursor.fetchall():
    print(f"{row[0]:6} | {row[1]:.2f}")

Неделя | Среднее количество событий
     6 | 13.80
     7 | 9.62
     8 | 8.03
     9 | 9.39
    10 | 8.21
    11 | 10.02
    12 | 9.38
    13 | 10.01
    14 | 9.86
    15 | 10.35
    16 | 10.29
    17 | 10.52
    18 | 9.67
    19 | 11.11
    20 | 14.45
    21 | 18.50
    22 | 22.49
    23 | 22.26
    24 | 23.01
    25 | 18.22
    26 | 8.60
    27 | 1.25
    28 | 0.09
    29 | 0.05


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [ ]:
cursor.execute("SELECT d.department_name, COUNT(DISTINCT l.userid) AS excellentCount FROM user_logs AS l " \
"JOIN departments d ON l.depart_id = d.id " \
"WHERE l.NameR_Level = '5' " \
"GROUP BY d.department_name " \
"ORDER BY excellentCount DESC " \
"LIMIT 1; " \
"")

dept_ex, cnt_ex = cursor.fetchone()
print(f"Больше всего отличников на кафедре '{dept_ex}' — {cnt_ex} чел.")


cursor.execute("SELECT d.department_name, COUNT(DISTINCT l.userid) AS excellentCount FROM user_logs AS l " \
"JOIN departments d ON l.depart_id = d.id " \
"WHERE l.NameR_Level = '2' " \
"GROUP BY d.department_name " \
"ORDER BY excellentCount DESC " \
"LIMIT 1; " \
"")

dept_bad, cnt_bad = cursor.fetchone()
print(f"Больше всего двоечников на кафедре '{dept_bad}' — {cnt_bad} чел.")


Больше всего отличников на кафедре 'ДиСО' — 310 чел.
Больше всего двоечников на кафедре 'Эконом.' — 72 чел.


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [ ]:
cursor.execute("" \
"WITH peak_week AS (" \
"SELECT num_week FROM user_logs GROUP BY num_week ORDER BY SUM(s_all) DESC LIMIT 1 ) " \
"SELECT l.Name_OsnO as name_osno, l.num_week as week_number, AVG(l.s_all) AS avg_s_all, AVG(l.s_course_viewed) AS avg_s_course_viewed, AVG(l.s_q_attempt_viewed) AS avg_s_q_attempt_viewed FROM user_logs l " \
"JOIN peak_week p ON l.num_week = p.num_week " \
"GROUP BY l.name_osno, l.num_week; " \
"") 


print("| name_osno | week_number | avg_s_all | avg_s_course_viewed | avg_s_q_attempt_viewed |")
for row in cursor.fetchall():
    # Если нужно заменить числовые коды на текст
    osno = row[0]
    if osno == '1' or osno == 1:
        osno = "бюджет"
    elif osno == '2' or osno == 2:
        osno = "контракт"
    print(f"| {osno} | {row[1]} | {row[2]:.2f} | {row[3]:.2f} | {row[4]:.2f} |")

| name_osno | week_number | avg_s_all | avg_s_course_viewed | avg_s_q_attempt_viewed |
| контракт | 24 | 28.69 | 4.73 | 5.42 |
| бюджет | 24 | 20.79 | 3.78 | 4.94 |


персональное задание

вывести среднюю оценку по курсу за которым закреплено наибольшее количество кафедр

In [ ]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
cursor.close()
conn.close()